<a href="https://colab.research.google.com/github/AhnafTouseef/Blender-Colab-render-pipline/blob/main/Rendering_pipeline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# @title # ***`File navigation commands`***
try:
  from google.colab import drive
  drive.mount('/content/drive')
except:
  pass

import requests

# Fetch the raw text of whatever script you need
script_url = "https://raw.githubusercontent.com/AhnafTouseef/colab-file-navigator/main/colab_tools.py"
remote_code = requests.get(script_url).text

# Run it immediately into the cell's memory
exec(remote_code)

Type "help()" for help


# ***`Blender Installation`***
---

In [12]:
import subprocess
from pathlib import Path
import re

comp = re.compile('blender')
check = []
for item in Path.iterdir(path):
    check.append(bool(comp.search(str(item))))

if True in check:
    print("Blender already installed")
else:
    subprocess.run(["wget", "https://download.blender.org/release/Blender4.5/blender-4.5.1-linux-x64.tar.xz"])
    subprocess.run(["tar", "xf", "blender-4.5.1-linux-x64.tar.xz"])
    delete('/content/blender-4.5.1-linux-x64.tar.xz')



Blender already installed


# ***`Configs`***
---

In [ ]:
import os

#@markdown ## 🏢 Step 1: Core Worker & Project Setup


TOTAL_WORKER = 5 # @param {type:"integer"}
RENDER_FILE = "barbershop_interior" # @param {type:"string"}

#@markdown ---
#@markdown ---
#@markdown ## ⚙️ Step 2: Render Engine & Performance

RENDER_ENGINE = "CYCLES" # @param ["CYCLES","BLENDER_EEVEE_NEXT","BLENDER_WORKBENCH"] {type:"string"}
RENDER_SAMPLES = 128 # @param {type:"integer"}
RESOLUTION_PERCENT = 50 # @param {type:"slider", min:0, max:100, step:1}


#@markdown ### 🎯 Adaptive Sampling Settings
ADAPTIVE_SAMPLING = True # @param {type:"boolean"}
ADAPTIVE_THRESHOLD = 0.1 # @param {type:"number"}

#@markdown ---
#@markdown ---
#@markdown ## 🧼 Step 3: Denoising & Post-Processing

USE_DENOISE = True # @param {type:"boolean"}
DENOISER = "OPENIMAGEDENOISE" # @param ["OPTIX", "OPENIMAGEDENOISE", "NONE"] {type:"string"}

#@markdown ---
#@markdown ---
#@markdown ## 🎬 Step 4: Output & Animation Settings

RENDER_ANIMATION = False # @param {type:"boolean"}
FPS = 24 # @param {type:"integer"}
OUTPUT_FORMAT = "PNG" # @param ["PNG", "JPEG", "OPEN_EXR", "FFMPEG"] {type:"string"}
USE_PERSISTENT_DATA = True # @param {type:"boolean"}
SPATIAL_SPLITS = True # @param {type:"boolean"}


# Directory for configuration file
CONFIG_DIR = "/content/drive/MyDrive/Render_Farm"

# Ensure the directory exists
# os.makedirs(CONFIG_DIR, exist_ok=True)

# Construct the Blender Python API content for render settings
render_settings_content = f"""
import bpy;
S = bpy.context.scene;
S.render.engine = '{RENDER_ENGINE}';
S.cycles.use_adaptive_sampling = {ADAPTIVE_SAMPLING};
S.cycles.adaptive_threshold = {ADAPTIVE_THRESHOLD};
S.cycles.samples = {RENDER_SAMPLES};
S.render.fps = {FPS};
S.render.resolution_percentage = {RESOLUTION_PERCENT};
S.cycles.use_denoising = {USE_DENOISE};
S.cycles.denoiser = '{DENOISER}';
S.render.use_persistent_data = {USE_PERSISTENT_DATA};
S.render.image_settings.file_format = '{OUTPUT_FORMAT}';
S.cycles.debug_use_spatial_splits = {SPATIAL_SPLITS};
"""

# Save the Blender API settings to a Python file
render_settings_file_path = os.path.join(CONFIG_DIR, 'render_settings.py')
with open(render_settings_file_path, 'w') as f:
    f.write(render_settings_content)

# Configure JSON file
with open(r'/content/drive/MyDrive/Render_Farm/config.json', 'w') as f:
  string = '{' + f'\n"total_workers" : {TOTAL_WORKER}, \n"blend_file" : "{RENDER_FILE+".blend"}"\n' + '}'
  f.write(string)

print(f"Blender render settings saved to {render_settings_file_path}")


Blender render settings saved to /content/drive/MyDrive/Render_Farm/render_settings.py


# ***`Render Setup`***
---

In [ ]:
# @title
import os
import sys
import json
import re
import subprocess
from IPython.display import clear_output

# ============================================================
# SETTINGS
# ============================================================

MAIN_DIRECTORY = "/content/drive/MyDrive/Render_Farm"

CONFIG_FILE = f"{MAIN_DIRECTORY}/config.json"
PYTHON_OVERRIDE = f"{MAIN_DIRECTORY}/render_settings.py"
REGISTER_DIR = f"{MAIN_DIRECTORY}/register"
OUTPUT_DIR = f"{MAIN_DIRECTORY}/output"

BLENDER_EXEC = "/content/blender-4.5.1-linux-x64/blender"

# Output settings
OUTPUT_FORMAT = "PNG"          # PNG / JPEG / OPEN_EXR
RESOLUTION_PERCENT = 20     # In percentage %
RENDER_SAMPLES = 1         # None = use blend settings


os.makedirs(REGISTER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)

TOTAL_WORKERS = int(config["total_workers"])
BLEND_PATH = f"{MAIN_DIRECTORY}/{config['blend_file']}"


# ============================================================
# WORKER
# ============================================================

def start_render():
    def register_worker():
        ids = [int(os.path.splitext(x)[0]) for x in os.listdir(REGISTER_DIR) if os.path.splitext(x)[0].isdigit()]
        worker = max(ids) + 1 if ids else 1

        if worker == TOTAL_WORKERS:
            print(f"{worker}th worker arived. Cleaning registry")
            for file in os.listdir(REGISTER_DIR):
                os.remove(f"{REGISTER_DIR}/{file}")
            return worker
        else:
            open(f"{REGISTER_DIR}/{worker}.txt", "w").close()
            print(f"Worker {worker} registered")
            return worker


    worker_id = register_worker()


    # ============================================================
    # BLENDER DATA
    # ============================================================

    def get_scene_data():
        code = "import bpy; c=bpy.context.scene; print(f'START:{c.frame_start}\\nEND:{c.frame_end}')"

        res = subprocess.run(
            f'{BLENDER_EXEC} -b {BLEND_PATH} --python-expr "{code}"',
            shell=True,
            text=True,
            capture_output=True,
        )

        # Extract numbers directly using split
        print(res.stdout)
        start = res.stdout.split("START:")[1].split("\n")[0]
        end = res.stdout.split("END:")[1].split("\n")[0]
        return {"start_frame":start, "end_frame":end}


    scene = get_scene_data()

    START_FRAME = int(scene["start_frame"])
    END_FRAME = int(scene["end_frame"])



    def split_frames(worker_index, total_workers, start_frame, end_frame):
        total_frame = end_frame - start_frame + 1
        division, remainder = divmod(total_frame, total_workers)
        size = division + (1 if worker_index < remainder else 0)
        offset = worker_index * division + min(worker_index, remainder)
        return start_frame + offset, start_frame + offset + size - 1


    my_start, my_end = split_frames(worker_id - 1, TOTAL_WORKERS, START_FRAME, END_FRAME)



    # ============================================================
    # BLENDER COMMAND
    # ============================================================


    cmd = [BLENDER_EXEC, "-b", BLEND_PATH,"-P", PYTHON_OVERRIDE,"-o", f"{OUTPUT_DIR}/Frame_","-s", str(my_start), "-e", str(my_end), "-a","--", "--cycles-device", "OPTIX"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    finder = re.compile(r"Fra:(\d+)")


    # ============================================================
    # LIVE MONITOR
    # ============================================================

    current_frame = my_start

    for line in process.stdout:
        # clear_output(wait=True)
        # print(line)

        try:
          frame = finder.search(line).group(1)
        except:
          continue
        # print(frame)
        if frame:
            current_frame = int(frame)

        frame_ratio = (current_frame - my_start + 1) / (my_end - my_start + 1)

        frame_bar = "█" * int(frame_ratio * 30) + "-" * (30 - int(frame_ratio * 30))

        clear_output(wait=True)
        print(f"Project name: {config['blend_file']}, Start Frame: {START_FRAME}, End Frame{END_FRAME}")

        print("=" * 50)
        print(f"Worker {worker_id}/{TOTAL_WORKERS}")
        print(f"Frames {my_start} -> {my_end}")
        print(f"Output {OUTPUT_DIR}")
        print("=" * 50)

        print("=" * 50)
        print("Blender Render Monitor")
        print("=" * 50)
        print(f"Frame   : {current_frame}/{my_end}")
        print(f"Overall : [{frame_bar}] {frame_ratio * 100:.2f}%")




    process.wait()


    if process.returncode == 0:
        print("\n✅ Render finished")
    else:
        print("\n❌ Render failed")

# ***`Monitor`***
---

In [ ]:
start_render()

Project name: barbershop_interior.blend, Start Frame: 1, End Frame5
Worker 2/5
Frames 2 -> 2
Output /content/drive/MyDrive/Render_Farm/output
Blender Render Monitor
Frame   : 2/2
Overall : [██████████████████████████████] 100.00%

✅ Render finished
